<style>
  .cell-markdown { overflow: auto !important; }
  .mermaid { max-width: 100%; height: auto; }
</style>

# Cumulative Windows Walkthrough


## Overview

[Preparation](#prep)

* [Topology](#topology)
* [Steps](#steps)


---
<a id="prep"></a>
## Preparation

Before we start off, we first prepare for the examples to follow:

In [2]:
import sys
sys.path.insert(1, "../..")
sys.path.insert(1, "../../../..")

from kafi.streams.topologynode import TopologyNode as Tn

from generators import OrderGenerator
order_generator = OrderGenerator()

order_source_str = "orders"
sink_str = "sink"

#

def process(built_tn, customer_id, price, ts, w=1):
    m = {"value": {"customer_id": customer_id, "price": price, "ts": ts}}
    #
    sink_str_r_list_dict = built_tn.process({order_source_str: [(m, w)]})
    r_list = sink_str_r_list_dict[sink_str]
    #
    print("Triggers:")
    for r in r_list:
        print(r)


<a id="topology"></a>
---
## Topology

Now it's time for the walkthrough itself. We go for a slightly simpler example for the walkthroughs.

The corresponding test can be found here: [test_windows.py](../../../../test/streams/test_windows.py)

In [8]:
import kafi.streams.topologynode
import importlib
importlib.reload(kafi.streams.topologynode)

from kafi.streams.topologynode import TopologyNode as Tn

size_int = 100
advance_int = size_int // 5
allowed_lateness_int = size_int * 2
#
order_source_tn = Tn.source(order_source_str)
order_source_tn.to_zSet(Tn._from_records)
order_tn = (
    order_source_tn
    # 1. Select customer_id, price and ts from the value.
    .map(lambda r: {"customer_id": r["value"]["customer_id"],
                    "price": r["value"]["price"],
                    "ts": r["value"]["ts"]})
    # 2. Expire with window size order_generator.ts_step_int * 100, advance size 100 / 5 = 20 and allowed_lateness = window_size * 2,  
    .expire_cumulative(lambda r: r["ts"], size_int, advance_int, allowed_lateness_int)
    # 3. Deduplicate.
    .distinct()
)
#
# 4. Set up the window: group by customer ID, count the orders, sum up the prices of the orders and get the last timestamp of the window.
sink_tn = order_tn.group_by_agg_cumulative(
    ts_fun=lambda r: r["ts"],
    size_int=size_int,
    advance_int=advance_int,
    key_fun=lambda r: r["customer_id"],
            agg_fun=lambda agg_r, r: {"orders": agg_r["orders"] + 1,
                                      "total_price": agg_r["total_price"] + r["price"],
                                      "last_ts": max(agg_r["last_ts"], r["ts"])},
            agg_initial_any={"orders": 0, "total_price": 0, "last_ts": 0},
            project_fun=lambda key_any, agg_r: {"customer_id": key_any,
                                                "orders": agg_r["orders"],
                                                "total_price": agg_r["total_price"],
                                                "last_ts": agg_r["last_ts"]},
    trigger_positive_only_bool=False
).sink(sink_str)
#
_ = built_tn = Tn.build(sink_tn)

What do we do?
1. `map()`: Select customer_id, price and ts from the value.
2. `expire_cumulative()`: Expire with window size `100`, advance size `100 // 5 = 20` and `allowed_lateness` = `size_int * 2 = 200`.  
3. `distinct()`: Deduplicate.
4. `group_by_agg_cumulative()`: Set up the window: group by customer ID, count the orders sum up the prices of the orders and get the last timestamp of the window.

Next, we illustrate how the tumbling window works by processing some example data - one by one, in baby steps.

We use two types of illustrations in each step:
1. Top-down view:
  * time proceeds from top to bottom (starting with `0`)
  * the start and end times of the time windows are marked by small grey circles
  * the latest timestamp of the input after the respective step is written at the top 
  * new events coming in a step are indicated a blue frame
  * old events have a grey frame
  * the triggered outputs in the sink are indicated by green color
2. Left-right view:
  * time proceeds from left to right (starting with `0`)
  * new windows appear below the old windows
  * `^`: latest timestamp of this step
  * `(^)`: latest timestamp of the previous step
  * `<<<`: time window(s) containing the event from this step
  * `!!!`: time window(s) triggered by the event from this step



<a id="steps"></a>
## Steps

### Step 1

In step 1, the first order from customer 1 arrives at timestamp 10:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 10"]
        direction TB
        0(("0")) e1@-.-> 10
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#1565c0,stroke-width:6px
```

In step 1, first, the first order arrives from customer 1 at timestamp 10:

This event:
* advances the latest timestamp from `0` (start, visualized by `(^)`) to `10` (visualized by `^`),
* and falls into the first cumulative windows `[0, 20)`, `[0, 40)`, `[0, 60)`, `[0, 80)`, `[0, 100)` (visualized by `<<<`):
```
[0 -- 20) <<<
[0 ---- 40) <<<
[0 ------ 60) <<<
[0 -------- 80) <<<
[0 ---------- 100) <<<
(^) 
   ^
```

As the latest timestamp is not yet beyond the end of the first cumulative window, no output is triggered. Why?

Because the default `trigger_fun` is defined as `lambda r_end_ts_tuple, latest_ts: latest_ts >= r_end_ts_tuple[1]` where `r_end_ts_tuple[1] = end_ts = 20` (`20` being the end of the first of the windows). At this point, `latest_ts = 10 >= 20` evaluates to `False` and consequently, no output is triggered.

Let's see this happening for real:

In [9]:
process(built_tn, customer_id=1, price=100, ts=10, w=1)


[20, 40, 60, 80, 100]
[20, 40, 60, 80, 100]
Triggers:


### Step 2

In baby step 2, the second order arrives from customer 1 at timestamp 30:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 30"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 30
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    30@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 30}"}
    style 30 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp from `10` to `30`,
* falls into the cumulative windows `[0, 40)`, `[0, 60)`, `[0, 80)`, `[0, 100)`
* and triggers the first cumulative 
```
[  0 --  20) <<<
[  0 ----  40) <<<
[  0 ------  60) <<<
[  0 --------  80) <<<
[  0 ---------- 100) <<<
[ 20 ------------ 120) <<<
(^) 
     ^
```


In [10]:
process(built_tn, customer_id=1, price=200, ts=30, w=1)



[40, 60, 80, 100]
[40, 60, 80, 100]
Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 100, 'last_ts': 10, 'window_end': 20}


### Step 3

In step 3, an order from customer arrives shortly after the end of the first tumbling window:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 300,\n&quot;last_ts&quot;: 50\n&quot;window_end&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    105 --- space1
    linkStyle 4 stroke:none
    105 Link@== Triggers ==> Output
    linkStyle 5 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp from `10` to `105`,
* falls into the second tumbling window `[100, 200)`,
* and triggers the first window `[0, 100)` (visualized to `!!!`):
```
[0 ---------- 100) !!!
             [100 ---------- 200) <<<
      (^) 
                  ^
```

Now the latest timestamp of the input stream *is* beyond the end of the first tumbling window.

Hence, the aggregation of the two orders from customer 1 that came in between `[0, 100)` is triggered, because `latest_ts = 105 >= 100` evaluates to `True` in `trigger_fun`:

In [6]:
process(built_tn, customer_id=2, price=50, ts=105, w=1)

Triggers:
{'customer_id': 1, 'orders': 2, 'total_price': 300, 'last_ts': 50, 'window_end': 100}


### Step 4

Step 4 shows the effect of a retraction coming in (weight = `-1`) as the order from customer 1 at timestamp `50` is canceled:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s style='color:red;'>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#bb0000,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 100,\n&quot;last_ts&quot;: 10\n&quot;window_end&quot;: 100}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    50 Link@== Triggers ==> Output
    linkStyle 4 stroke:#00bb00,stroke-width:3px;
```

This event:
* does not advance the latest timestamp because its weight is `-1` (the latest timestamp stays at `105`),
* falls into the tumbling window `[100, 200)`,
* and triggers the correction of the first window `[0, 100)`:
```
[0 ---------- 100) !!!
             [100 ---------- 200) <<<
                 (^) 
                  ^
```

The correction still comes in early enough as in still within the `allowed_lateness = 200` (actually, it comes in during the *window buffer time* already): `105 (latest) - ( 100 (window end for ts = 50) + 100 (window buffer) = 200 ) = -95 < 200`

As a result, the corresponding correction of the tumbling window `[0, 100)` is triggered.

In [7]:
process(built_tn, customer_id=1, price=200, ts=50, w=-1)


Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 100, 'last_ts': 10, 'window_end': 100}


### Step 5

An order from customer 3 arrives out of order at timestamp `101`:

```mermaid
flowchart TB
    subgraph x_axis ["Latest timestamp = 105"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* does not advance the latest timestamp (it stays at `105`),
* falls into the tumbling window `[100, 200)`,
* and triggers nothing:
```
[0 ---------- 100)
             [100 ---------- 200) <<<
                 (^) 
                  ^
```

The tumbling window `[0, 100)` for customer 1 hasn't changed as the order from customer 3 lands in the tumbling windows for `[100, 200)`.


In [8]:
process(built_tn, customer_id=3, price=400, ts=101, w=1)

Triggers:


### Step 6

Another order from customer 3 arrives at timestamp `150`:

```mermaid
flowchart LR
    subgraph x_axis ["Latest timestamp = 150"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 150
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* advances the latest timestamp to `150`,
* falls into the tumbling window `[100, 200)`,
* and still triggers nothing:
```
[0 ---------- 100)
             [100 ---------- 200) <<<
                 (^) 
                      ^
```

We are still not beyond the second tumbling window `[100, 200)`. No output.


In [9]:
process(built_tn, customer_id=3, price=200, ts=150, w=1)


Triggers:


### Step 7

Customer 1 creates another order at timestamp 210:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 210"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 150 e7@-.-> 200(("200")) e8@-.-> 210
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#1565c0,stroke-width:6px

    Output1@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50\n&quot;last_ts&quot;: 105,\n&quot;window_end&quot;: 200}"}
    style Output1 fill:none,stroke:#00bb00,stroke-width:6px
    210 Link@== Triggers ==> Output1
    linkStyle 8 stroke:#00bb00,stroke-width:3px;

    Output2@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 600,\n&quot;last_ts&quot;: 150,\n&quot;window_end&quot;: 200}"}
    style Output2 fill:none,stroke:#00bb00,stroke-width:6px
    210 Link@== Triggers ==> Output2
    linkStyle 9 stroke:#00bb00,stroke-width:3px;
```

This event:
* advances the latest timestamp to `210`,
* falls into the tumbling window `[200, 300)`,
* and triggers the passed window `[100, 200)`:
```
[0 ---------- 100)
             [100 ---------- 200) !!!
                            [200 ---------- 300) <<<
                     (^) 
                                 ^
```


In [10]:
process(built_tn, customer_id=1, price=50, ts=210, w=1)

Triggers:
{'customer_id': 3, 'orders': 2, 'total_price': 600, 'last_ts': 150, 'window_end': 200}
{'customer_id': 2, 'orders': 1, 'total_price': 50, 'last_ts': 105, 'window_end': 200}


### Step 8

An order from customer 2 arrives at timestamp `330`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 330"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 150 e7@-.-> 200(("200")) e8@-.-> 210 e9@-.-> 300(("300")) e10@-.-> 330
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 50\n&quot;last_ts&quot;: 210,\n&quot;window_end&quot;: 300}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    330 --- space1
    linkStyle 10 stroke:none
    330 Link@== Triggers ==> Output
    linkStyle 11 stroke:#00bb00,stroke-width:3px
```

This event:
* advances the latest timestamp to `330`,
* falls into the tumbling window `[300, 400)`,
* and triggers the passed window `[200, 300)`:
```
[0 ---------- 100)
             [100 ---------- 200)
                            [200 ---------- 300) !!!
                                           [300 ---------- 400) <<<
                                (^) 
                                                  ^
```


In [11]:
process(built_tn, customer_id=2, price=60, ts=330, w=1)

Triggers:
{'customer_id': 1, 'orders': 1, 'total_price': 50, 'last_ts': 210, 'window_end': 300}


### Step 9

An order from customer 2 arrives late but not too late, at timestamp `120`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 330"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 180 e7@-.-> 150 e8@-.-> 200(("200")) e9@-.-> 210 e10@-.-> 300(("300")) e11@-.-> 330
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    180@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 40,\n&quot;ts&quot;: 180}"}
    style 180 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 2,\n&quot;total_price&quot;: 90\n&quot;last_ts&quot;: 180,\n&quot;window_end&quot;: 200}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    180 Link@== Triggers ==> Output
    linkStyle 11 stroke:#00bb00,stroke-width:3px
```

This event:
* does not advance the latest timestamp (it stays at `330`),
* falls into the tumbling window `[100, 200)`,
* and triggers the correction of the passed window `[100, 200)` (because it is contained in it):
```
[0 ---------- 100)
             [100 ---------- 200) <<< !!!
                            [200 ---------- 300)
                                           [300 ---------- 400)
                                                 (^) 
                                                  ^
```

This late arrival is still within the `allowed_lateness = 200`: `330 (latest) - ( 200 (window end for ts = 120) + 100 (window buffer) = 300 ) = 30 < 200`.


In [12]:
process(built_tn, customer_id=2, price=40, ts=180, w=1)

Triggers:
{'customer_id': 2, 'orders': 2, 'total_price': 90, 'last_ts': 180, 'window_end': 200}


### Step 10

An order from customer 1 arrives at timestamp `710`:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 710"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 180 e7@-.-> 150 e8@-.-> 200(("200")) e9@-.-> 210 e10@-.-> 300(("300")) e11@-.-> 330 e12@-.-> 400(("400")) e13@-.-> 500(("500")) e14@-.-> 710
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }
    e12@{ animate: true }
    e13@{ animate: true }
    e14@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    180@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 40,\n&quot;ts&quot;: 180}"}
    style 180 fill:none,stroke:#333,stroke-width:6px

    710@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 70,\n&quot;ts&quot;: 710}"}
    style 710 fill:none,stroke:#1565c0,stroke-width:6px

    Output@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;orders&quot;: 1,\n&quot;total_price&quot;: 60\n&quot;last_ts&quot;: 330,\n&quot;window_end&quot;: 400}"}
    style Output fill:none,stroke:#00bb00,stroke-width:6px
    space1[ ]
    style space1 fill:none,stroke:none
    710 --- space1
    linkStyle 14 stroke:none
    710 Link@== Triggers ==> Output
    linkStyle 15 stroke:#00bb00,stroke-width:3px
```
This event:
* advances the latest timestamp to `710`,
* falls into the tumbling window `[700, 800)`,
* and triggers nothing:
```
[0 ---------- 100)
             [100 ---------- 200)
                            [200 ---------- 300)
                                           [300 ---------- 400)
                                                          [400 ---------- 500)
                                                                         [500 ---------- 600)
                                                                                        [600 ---------- 700)
                                                                                                       [700 ---------- 800) <<<
                                                 (^) 
                                                                                                            ^
```

This pushes the latest timestamp to `710`. No output is triggered.


In [14]:
process(built_tn, customer_id=1, price=70, ts=710, w=1)

Triggers:


But stop! 

This looks innocent at first but under the covers, the bad thing happening is that the topology has quietly dropped the aggregation in window `[300, 400)`.

The problem here is that the interval between the last two messages was "too long" : `710 (latest) - ( 400 (window end for ts = 330) + 100 (window buffer) = 500 ) = 210 > 200` and caused the `expire_tumbling()` operator to discard the order of customer 2 at timestamp `330`.

The workarounds described in [Watermarks](#watermarks) would, in this case, look like this:
* increase `allowed_lateness` (to fix only this example, it would have to be at least `210`)
* augment the source with artificial explicit "watermarks". The test `test_tumbling_explicit_watermarks()` in [test_windows.py](../../../../test/streams/test_windows.py) shows a way how to accommodate this (you need to slightly adapt the aggregation and the trigger function to cater for the artificial explicit watermarks).

### Step 11

A last order from customer 1 arrives, but this time too late at timestamp 130:

```mermaid
flowchart TD
    subgraph x_axis ["Latest timestamp = 710"]
        direction TB
        0(("0")) e1@-.-> 10 e2@-.-> 50 e3@-.-> 100(("100")) e4@-.-> 101 e5@-.-> 105 e6@-.-> 110 e7@-.-> 130 e8@-.-> 180 e9@-.-> 150 e10@-.-> 200(("200")) e11@-.-> 210 e12@-.-> 300(("300")) e13@-.-> 330 e14@-.-> 400(("400")) e15@-.-> 500(("500")) e16@-.-> 710
    end

    style x_axis fill:none,stroke:none
    style 0 fill:none,stroke:#333,stroke-width:2px
    style 100 fill:none,stroke:#333,stroke-width:2px
    style 200 fill:none,stroke:#333,stroke-width:2px
    style 300 fill:none,stroke:#333,stroke-width:2px
    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
    e4@{ animate: true }
    e5@{ animate: true }
    e6@{ animate: true }
    e7@{ animate: true }
    e8@{ animate: true }
    e9@{ animate: true }
    e10@{ animate: true }
    e11@{ animate: true }
    e12@{ animate: true }
    e13@{ animate: true }
    e14@{ animate: true }
    e15@{ animate: true }
    e16@{ animate: true }

    10@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 100,\n&quot;ts&quot;: 10}"}
    style 10 fill:none,stroke:#333,stroke-width:6px

    50@{ shape: lean-r, label: "<s>{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 50}</s>"}
    style 50 fill:none,stroke:#333,stroke-width:6px

    105@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 105}"}
    style 105 fill:none,stroke:#333,stroke-width:6px

    101@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 400,\n&quot;ts&quot;: 101}"}
    style 101 fill:none,stroke:#333,stroke-width:6px

    150@{ shape: lean-r, label: "{&quot;customer_id&quot;: 3,\n&quot;price&quot;: 200,\n&quot;ts&quot;: 150}"}
    style 150 fill:none,stroke:#333,stroke-width:6px

    210@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 50,\n&quot;ts&quot;: 210}"}
    style 210 fill:none,stroke:#333,stroke-width:6px

    330@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 60,\n&quot;ts&quot;: 330}"}
    style 330 fill:none,stroke:#333,stroke-width:6px

    180@{ shape: lean-r, label: "{&quot;customer_id&quot;: 2,\n&quot;price&quot;: 40,\n&quot;ts&quot;: 180}"}
    style 180 fill:none,stroke:#333,stroke-width:6px

    710@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 70,\n&quot;ts&quot;: 710}"}
    style 710 fill:none,stroke:#333,stroke-width:6px

    130@{ shape: lean-r, label: "{&quot;customer_id&quot;: 1,\n&quot;price&quot;: 70,\n&quot;ts&quot;: 130}"}
    style 130 fill:none,stroke:#1565c0,stroke-width:6px
```

This event:
* does not advance the latest timestamp (it stays at `710`),
* falls into the tumbling window `[100, 200)`,
* and (correctly) triggers nothing:
```
[0 ---------- 100)
             [100 ---------- 200) <<<
                            [200 ---------- 300)
                                           [300 ---------- 400)
                                                          [400 ---------- 500)
                                                                         [500 ---------- 600)
                                                                                        [600 ---------- 700)
                                                                                                       [700 ---------- 800)
                                                                                                           (^) 
                                                                                                            ^
```

Why is this event "too late"? Because it does not arrive inside our `allowed_lateness = 200`: `710 (latest) - ( 200 (window end for ts = 120) + 100 (window buffer) = 300 ) = 410 > 200`


In [15]:
process(built_tn, customer_id=1, price=70, ts=130, w=1)

Triggers:
